In [4]:
import pandas as pd

#INPUT = pd.read_csv("1788253047626_WVS_Cross-National_Wave_7_inverted_csv_v6_0.csv")

# 14 countries spanning distinct WVS/Inglehart-Welzel cultural zones
COUNTRIES = {
    "USA": "English-speaking",
    "GBR": "English-speaking",
    "DEU": "Protestant Europe",
    "NLD": "Protestant Europe",
    "RUS": "Orthodox Europe",
    "UKR": "Orthodox Europe",
    "MEX": "Latin America",
    "ARG": "Latin America",
    "CHN": "Confucian",
    "JPN": "Confucian",
    "KOR": "Confucian",
    "EGY": "African-Islamic",
    "NGA": "African-Islamic",
    "IDN": "South Asia / Islamic",
    "IND": "South Asian"
}

DIMENSIONS = ["SACSECVAL", "RESEMAVAL"]  # secular values, emancipative values

cols = ["B_COUNTRY_ALPHA", "W_WEIGHT"] + DIMENSIONS
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/AdvanceTopics/AdvanceTopics - Evaluating the Robustness of Cultural Prompting in Large Language Models Against Human Cultural Value Distributions/WVS_Cross-National_Wave_7_inverted_csv_v6_0.csv", usecols=cols)
df = df[df["B_COUNTRY_ALPHA"].isin(COUNTRIES)].copy()

for d in DIMENSIONS:
    df[d] = pd.to_numeric(df[d], errors="coerce")
    df.loc[df[d] < -1, d] = pd.NA  # WVS missing-data codes

def weighted_mean(group, col):
    g = group.dropna(subset=[col])
    if g.empty:
        return pd.NA
    return (g[col] * g["W_WEIGHT"]).sum() / g["W_WEIGHT"].sum()

rows = []
for country, group in df.groupby("B_COUNTRY_ALPHA"):
    row = {"country": country, "cultural_zone": COUNTRIES[country], "n": len(group)}
    for d in DIMENSIONS:
        row[d] = round(weighted_mean(group, d), 4)
    rows.append(row)

out = pd.DataFrame(rows).sort_values("cultural_zone")
out.to_csv("/content/drive/MyDrive/Colab Notebooks/AdvanceTopics/AdvanceTopics - Evaluating the Robustness of Cultural Prompting in Large Language Models Against Human Cultural Value Distributions/human_baseline_wvs.csv", index=False)
print(out.to_string(index=False))

country        cultural_zone    n  SACSECVAL  RESEMAVAL
    EGY      African-Islamic 1200     0.1741     0.2366
    NGA      African-Islamic 1237     0.2577     0.2875
    CHN            Confucian 3036     0.4013     0.3918
    JPN            Confucian 1353     0.4391     0.5563
    KOR            Confucian 1245     0.5639     0.4535
    GBR     English-speaking 2609     0.4259     0.6499
    USA     English-speaking 2596     0.3806     0.5605
    ARG        Latin America 1003     0.4207     0.4671
    MEX        Latin America 1741     0.4284     0.4382
    RUS      Orthodox Europe 1810     0.4464     0.3983
    UKR      Orthodox Europe 1289     0.4700     0.3996
    DEU    Protestant Europe 1528     0.3820     0.6759
    NLD    Protestant Europe 2145     0.4559     0.6601
    IDN South Asia / Islamic 3200     0.2581     0.3288
    IND          South Asian 1692     0.2640     0.3748


In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/AdvanceTopics/AdvanceTopics - Evaluating the Robustness of Cultural Prompting in Large Language Models Against Human Cultural Value Distributions/human_baseline_wvs.csv")

def farthest_point_sample(df, n, cols=("SACSECVAL", "RESEMAVAL")):
    pts = df[list(cols)].to_numpy()
    chosen = [int(np.argmax(np.linalg.norm(pts - pts.mean(0), axis=1)))]  # start furthest from centroid
    for _ in range(n - 1):
        dists = np.min([np.linalg.norm(pts - pts[c], axis=1) for c in chosen], axis=0)
        dists[chosen] = -1
        chosen.append(int(np.argmax(dists)))
    return df.iloc[chosen]

pilot = farthest_point_sample(df, n=15)
print(pilot[["country", "cultural_zone", "SACSECVAL", "RESEMAVAL"]].to_string(index=False))
pilot.to_csv("/content/drive/MyDrive/Colab Notebooks/AdvanceTopics/AdvanceTopics - Evaluating the Robustness of Cultural Prompting in Large Language Models Against Human Cultural Value Distributions/pilot_countries.csv", index=False)

country        cultural_zone  SACSECVAL  RESEMAVAL
    EGY      African-Islamic     0.1741     0.2366
    NLD    Protestant Europe     0.4559     0.6601
    CHN            Confucian     0.4013     0.3918
    KOR            Confucian     0.5639     0.4535
    IND          South Asian     0.2640     0.3748
    USA     English-speaking     0.3806     0.5605
    NGA      African-Islamic     0.2577     0.2875
    ARG        Latin America     0.4207     0.4671
    DEU    Protestant Europe     0.3820     0.6759
    UKR      Orthodox Europe     0.4700     0.3996
    JPN            Confucian     0.4391     0.5563
    IDN South Asia / Islamic     0.2581     0.3288
    GBR     English-speaking     0.4259     0.6499
    MEX        Latin America     0.4284     0.4382
    RUS      Orthodox Europe     0.4464     0.3983


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
